In [68]:
# !pip install "Transformers[Torch]"

In [3]:
import transformers
print(transformers.__version__)

5.23.0


In [2]:
import pandas as pd 
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

In [69]:
train_data = pd.read_csv("dataset/samsum-train.csv")
valid_data = pd.read_csv("dataset/samsum-validation.csv")
test_data = pd.read_csv("dataset/samsum-test.csv")

In [8]:
train_data.info()
train_data.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14732 entries, 0 to 14731
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        14732 non-null  object
 1   dialogue  14731 non-null  object
 2   summary   14732 non-null  object
dtypes: object(3)
memory usage: 345.4+ KB


(14732, 3)

In [9]:
train_data['dialogue'][0]

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [70]:
# Random Sampling 
train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
valid_data = valid_data.sample(n=500, random_state=42).reset_index(drop=True)

In [71]:
train_data

,id,dialogue,summary
0,13811908,Violet: hi! i came across this Austin's articl...,Violet sent Claire Austin's article.
1,13716431,Pat: So does anyone know when the stream is go...,Pat and Lou are waiting for The stream but Kev...
2,13810214,Jane: <gif_file>\r\nJane: Whaddya think? \r\nS...,Jane is updating her Tinder profile tonight an...
3,13729823,"Adam: Do u have a map of Paris?\r\nTom: Yes, W...",Tom has a map of Paris.
4,13681400,"Frank: Hi, how's the family?\r\nMike: great! S...","Mike is happy, because Sam's moved out. Mike a..."
...,...,...,...
3995,13681041,Barry: hello buddy\r\nMichael: hey\r\nBarry: d...,Barry and Michael will watch football instead ...
3996,13818705,Karen: Hey Lisa. Larissa and me have recently ...,Karen and Larissa moved to Belgium and ask Lis...
3997,13821859,"Miles: Hey, guys, I'm so sorry, but I missed t...","Miles has missed the bus, so he may be 15 minu..."
3998,13812716,Emma: did you finish the book I gave you?\r\nL...,"Emma gave ""The First Fifteen Lives of Harry Au..."


## Data Preprocessing 

In [72]:
import re

def clean_data(text):
    text = re.sub(r"\r\n", " ",text)            # remove nextline character
    text = re.sub(r"\s+", " ",text)             # extra spaces 
    text = re.sub(r"<.*?>", " ",text)           # remove html tags 
    text = re.sub(r"[^A-Za-z0-9\s]", "", text)  # remove punctuations
    text = text.strip().lower()                 # trailing space & lowercase 
    return text

In [73]:
train_data.columns

Index(['id', 'dialogue', 'summary'], dtype='object')

In [74]:
train_data['dialogue'] = train_data['dialogue'].apply(clean_data)
train_data['summary'] = train_data['summary'].apply(clean_data)

valid_data['dialogue'] = valid_data['dialogue'].apply(clean_data)
valid_data['summary'] = valid_data['summary'].apply(clean_data)

In [75]:
train_data['dialogue'][0]

'violet hi i came across this austins article and i thought that you might find it interesting violet   claire hi  thanks but ive already read it  claire but thanks for thinking about me'

## Tokenize

In [76]:
tokenizer = T5Tokenizer.from_pretrained('t5-small')

In [77]:
def tokenize(data):
    inputs = tokenizer(data['dialogue'],padding="max_length", max_length=512, truncation=True)
    targets = tokenizer(data['summary'],padding="max_length", max_length=128, truncation=True)

    inputs['labels'] = targets['input_ids']   # tokenizer return {'input_ids','attention_mask'}
    return inputs 

In [78]:
train_dataset = train_data.apply(tokenize, axis = 1).tolist()
valid_dataset = valid_data.apply(tokenize, axis = 1).tolist()

In [79]:
# train_dataset[0]

## working with model 

In [80]:
model = T5ForConditionalGeneration.from_pretrained('t5-small')

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [81]:
import torch

if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else :
    device = torch.device('cpu')

print('device :' , device)
model.to(device)

device : mps


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [82]:
# training arguments 

training_args = TrainingArguments(
    output_dir='./results',

    num_train_epochs= 6,
    weight_decay= 0.01,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,  # effective batch size = 2 * 4 = 8

    eval_strategy= 'epoch',
    save_strategy= 'epoch',

    warmup_steps= 500
)

In [83]:
trainer = Trainer(
    model = model,
    args= training_args,
    train_dataset= train_dataset,
    eval_dataset= valid_dataset  
)

In [85]:
# train the model 

trainer.train()

In [ ]:
model.save_pretrained('./saved_model')      # save model
tokenizer.save_pretrained('./saved_model')  # save tokenizer


In [ ]:
# load model and tokenizer 

model = T5ForConditionalGeneration.from_pretrained('./saved_model')
tokenizer = T5Tokenizer.from_pretrained('./saved_model')

In [87]:
# test the core logic for text summary 

def summary(dialogue):
  dialogue = clean_data(dialogue)     # clean data

  # tokenize 
  inputs = tokenizer(
      dialogue, 
      padding = 'max_length',
      max_length = 512,
      truncation = True,
      return_tensors = 'pt'
  ).to(device)

  # generate summary => return token ids
  model.to(device)
  targets = model.generate(
      input_ids = inputs['input_ids'],
      attention_mask = inputs['attention_mask'],
      max_length = 128,
      num_beams = 4,        # generate 4 output sequence and return best of them 
      early_stopping = True   # when 4 output produced then stop
  )

  # convert token ids to word summary
  summary = tokenizer.decode(targets[0], skip_special_tokens=True)  #skip start and eos 
  return summary 

In [ ]:
test_dialogue = """ 
sajid : how are you ? nirob 
nirob : i'm fine and you ?
sajid : i'm also fine . tell me about the current AI trends 
nirob : currently AI has became so enhanced that developer are scared that their jobs are in danger. many SAAS and product based big comapany's started to layoff developer because of advacement of AI. which is low cost and highly productive 
sajid : that is a terrible news nirob . what will happen if this continue ? isn't gobal economy will be collapse ?
nirob : nobody is gonna tell what to come yet. but experts are scared because of this advancement of AI
sajid : take care of you , and prepare yourself for this threats
nirob : you too.
"""

summ = summary(test_dialogue)

print("Summary: ", summ)